# Project 3 — RAG Pipeline (No LangChain)
### LLM Interview Study Assistant
---
##What I'm experimenting with

In this notebook, I'm building a basic RAG pipeline without using LangChain, so I can see the individual steps of the process directly.

The pipeline covers:

* Loading a text document
* Splitting the document into smaller chunks
* Creating embeddings for each chunk
* Storing the embeddings in a FAISS index
* Retrieving the most relevant chunks for a question
* Passing the retrieved context to TinyLlama
* Generating an answer using the retrieved context

The goal here is not to build another complete RAG application. I already have FitRag for that. This notebook is mainly useful for understanding what happens inside a basic RAG pipeline before using higher-level frameworks such as LangChain.

Experiment focus

I'm particularly looking at the relationship between:

document → chunks → embeddings → retrieval → context → generation

## Cell 1 — Installing library

- **sentence-transformers** — converts text into vectors
- **faiss-cpu** — stores and searches vectors

No LangChain, no version conflicts.

In [ ]:
!pip install -q sentence-transformers faiss-cpu
print("Installation complete!")

Installation complete!


## Cell 2 — Import everything needed

- **SentenceTransformer** — the embedding model
- **faiss** — vector database
- **numpy** — handles number arrays (vectors are arrays of numbers)
- **transformers** — load TinyLlama, same as fine-tuning projects
- **torch** — PyTorch, same as fine-tuning projects

In [ ]:
# SentenceTransformer converts any text into a vector of numbers
from sentence_transformers import SentenceTransformer

# faiss is the vector database that stores and searches our vectors
import faiss

# numpy handles arrays of numbers — vectors are stored as numpy arrays
import numpy as np

# These are the same imports you used in both fine-tuning projects
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
import torch

print("All imports successful!")
print("GPU available:", "YES —", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO — go to Runtime > Change Runtime Type > T4 GPU")

All imports successful!
GPU available: YES — Tesla T4


## Cell 3 — Read and split the document

Step 1 and 2 of RAG pipeline.

In [ ]:
# ---- STEP 1: READ THE DOCUMENT ----

with open('llm_interview_knowledge.txt', 'r', encoding='utf-8') as f:
    full_text = f.read()

print(f"Document loaded!")
print(f"Total characters: {len(full_text)}")
print(f"\nFirst 300 characters:")
print("-" * 50)
print(full_text[:300])
print("-" * 50)

Document loaded!
Total characters: 22811

First 300 characters:
--------------------------------------------------
LLM DEVELOPER INTERVIEW KNOWLEDGE BASE
Prepared for: Shruti Agrawal | Role: Accenture LLM Developer

SECTION 1: LARGE LANGUAGE MODELS - FUNDAMENTALS
------------------------------------------------

Q: What is a Large Language Model?
A:
--------------------------------------------------


In [ ]:
# ---- STEP 2: SPLIT INTO CHUNKS ----

raw_chunks = full_text.split('\n\n')

# Now we clean each chunk:
chunks = [chunk.strip() for chunk in raw_chunks if len(chunk.strip()) > 50]

print(f"Document split into {len(chunks)} chunks")
print(f"\nExample — Chunk 1:")
print("-" * 50)
print(chunks[0])
print("-" * 50)
print(f"\nExample — Chunk 5:")
print("-" * 50)
print(chunks[4])
print("-" * 50)

Document split into 44 chunks

Example — Chunk 1:
--------------------------------------------------
LLM DEVELOPER INTERVIEW KNOWLEDGE BASE
Prepared for: Shruti Agrawal | Role: Accenture LLM Developer
--------------------------------------------------

Example — Chunk 5:
--------------------------------------------------
Q: What is the Attention mechanism?
A: Attention is the core innovation of the Transformer. It allows the model to look at all words in a sentence at the same time and decide which words are most relevant to understanding each other word. For each word the model creates three vectors called Query, Key, and Value. The Query asks "what am I looking for?", the Key says "what do I contain?", and the Value is the actual information passed forward. The model computes how well each Query matches each Key, and the best matching Keys contribute their Values to the final representation. This is called Self-Attention.
--------------------------------------------------


## Cell 4 — Create embeddings and build vector store

This cell covers the embedding and vector-search part of the RAG pipeline.

Embeddings

Each text chunk is converted into a numerical vector that represents its semantic meaning.

The embedding model used here produces vectors with 384 dimensions. Chunks with similar meaning should generally have vectors that are closer together in the embedding space.

FAISS index

FAISS stores these vectors and allows similarity searches over them.

When a question is asked:

question → embedding → similarity search → relevant chunks

The retrieved chunks are then passed to the language model as context for generating the answer.

The important part I'm observing here is that retrieval and generation are separate steps. FAISS is responsible for finding relevant information; it does not generate the final answer.

In [ ]:
# ---- STEP 3: LOAD EMBEDDING MODEL ----

print("Loading embedding model...")
embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print("Embedding model loaded!")

# ---- CONVERT ALL CHUNKS TO VECTORS ----

print("\nConverting all chunks to vectors...")
chunk_embeddings = embed_model.encode(
    chunks,
    show_progress_bar=True
)

print(f"\nEmbeddings created!")
print(f"Shape: {chunk_embeddings.shape}")
# shape tells us: (number of chunks, vector size)

# ---- STEP 4: BUILD FAISS INDEX ----

EMBEDDING_DIM = 384

index = faiss.IndexFlatL2(EMBEDDING_DIM)

index.add(np.array(chunk_embeddings).astype('float32'))

print(f"\nFAISS index built!")
print(f"Total vectors stored in index: {index.ntotal}")
print("RAG database is ready to search!")

Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded!

Converting all chunks to vectors...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


Embeddings created!
Shape: (44, 384)

FAISS index built!
Total vectors stored in index: 44
RAG database is ready to search!


## Cell 5 — Test retrieval before adding the LLM

Very important step — always test retrieval separately first.

This helps check whether the vector search is returning relevant chunks for the question.

If the retrieved context is not relevant, the final generated answer is unlikely to be useful, so it's important to inspect retrieval before moving on to generation.

This cell does not use the LLM — it only searches the vector store and displays the retrieved chunks.

In [ ]:
def retrieve_chunks(question, k=3):
    """
    Find the k most relevant chunks for a given question.

    What happens inside:
    1. Convert the question to a vector using the same embedding model
    2. Ask FAISS to find the k closest vectors in the index
    3. Return the text chunks at those positions

    question: the text question to search for
    k: how many chunks to return (default 3)
    """

    # Step 1: Convert question to vector
    question_vector = embed_model.encode([question]).astype('float32')

    # Step 2: Search FAISS for the k nearest vectors
    distances, indices = index.search(question_vector, k)

    # Step 3: Get the actual text chunks using the position numbers
    relevant_chunks = [chunks[i] for i in indices[0]]

    return relevant_chunks, distances[0]


# Testing retrieval with a question
test_question = "What is LoRA and how does it work?"
results, scores = retrieve_chunks(test_question, k=3)

print(f"Question: '{test_question}'")
print(f"\nTop {len(results)} most relevant chunks found:")
print("=" * 60)

for i, (chunk, score) in enumerate(zip(results, scores)):

    print(f"\nChunk {i+1} (distance score: {score:.3f} — lower is better):")
    print("-" * 40)
    print(chunk[:400])
    if len(chunk) > 400:
        print("...")

print("\n" + "=" * 60)
print("If the chunks above are relevant to the question — retrieval is working!")

Question: 'What is LoRA and how does it work?'

Top 3 most relevant chunks found:

Chunk 1 (distance score: 0.444 — lower is better):
----------------------------------------
Q: What is LoRA?
A: LoRA stands for Low-Rank Adaptation. It is a parameter-efficient fine-tuning method that avoids updating all model parameters. Instead of modifying the original model weights, LoRA freezes the entire pre-trained model and adds pairs of small trainable matrices alongside specific layers. During training only these small matrices are updated. After training their output is added 
...

Chunk 2 (distance score: 0.917 — lower is better):
----------------------------------------
Q: What are LoRA hyperparameters?
A: The key LoRA hyperparameters are rank (r), lora_alpha, lora_dropout, and target_modules. Rank (r) controls the size of the LoRA matrices - higher rank means more parameters and potentially better learning but more memory. Common values are 8, 16, or 32. lora_alpha is a scaling factor that 

## Cell 6 — Load TinyLlama

Exactly same the fine-tuning projects.
The only difference — we use pipeline() for easier inference
instead of manually calling model.generate().

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Same 4-bit quantization config as fine-tuning projects
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading TinyLlama...")

# Load model — same as fine-tuning projects
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# Load tokenizer — same as fine-tuning projects
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# temperature=0.3: low temperature for focused factual answers
# do_sample=True: use probability sampling not just top token
# repetition_penalty=1.3: discourages repeating the same phrases
llm_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    temperature=0.3,
    do_sample=True,
    repetition_penalty=1.3
)

print("TinyLlama ready!")

Loading TinyLlama...


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'do_sample', 'repetition_penalty', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


TinyLlama ready!


## Cell 7 — Build the complete RAG function

This is the heart of the project.
One function that runs the entire RAG pipeline end to end:
question → retrieve chunks → build prompt → LLM → answer


In [ ]:
def ask_rag(question, k=3):
    """
    Full RAG pipeline in one function.

    Steps:
    1. Retrieve k most relevant chunks for the question
    2. Join chunks into one context string
    3. Build a prompt with context + question
    4. Send prompt to TinyLlama
    5. Extract and return clean answer
    """

    # ---- STEP 5: RETRIEVE RELEVANT CHUNKS ----
    relevant_chunks, scores = retrieve_chunks(question, k=k)

    context = '\n\n'.join(relevant_chunks)

    # ---- STEP 6: BUILD THE PROMPT ----
    prompt = f"""<|system|>
You are an expert LLM Developer interview coach.
Answer the question using ONLY the context provided below.
Be clear and detailed. If the answer is not in the context,
say 'This topic is not in the knowledge base.'</s>
<|user|>
Context from knowledge base:
{context}

Question: {question}</s>
<|assistant|>
"""

    # ---- STEP 7: GENERATE ANSWER ----
    raw_output = llm_pipeline(prompt)[0]['generated_text']

    # ---- CLEAN THE OUTPUT ----
    if '<|assistant|>' in raw_output:
        answer = raw_output.split('<|assistant|>')[-1].strip()
    else:
        answer = raw_output.strip()

    return answer, relevant_chunks


print("RAG function ready!")
print("Full pipeline: Question → Retrieve → Prompt → LLM → Answer")

RAG function ready!
Full pipeline: Question → Retrieve → Prompt → LLM → Answer


## Cell 8 — Test the Full RAG Pipeline
Now I'm testing the complete RAG pipeline with several questions from the knowledge document.

For each question, the pipeline:

- retrieves relevant chunks from the vector store
- passes the retrieved context to the LLM
- generates an answer
- displays the source chunks used for retrieval

I'm checking both the generated response and the retrieved sources to see how well the pipeline is grounding its answers.

In [ ]:
# testing on real interview questions to test the RAG pipeline
interview_questions = [
    "What is LoRA and why is it used?",
    "What is the difference between pre-training and fine-tuning?",
    "What is RAG and why is it important?",
]

for question in interview_questions:
    print("\n" + "=" * 65)
    print(f"QUESTION: {question}")
    print("=" * 65)

    answer, sources = ask_rag(question)

    print(f"\nANSWER:\n{answer}")

    print(f"\nSOURCE CHUNKS USED:")
    for i, source in enumerate(sources):
        # Show only first 120 characters of each source chunk
        print(f"  [{i+1}] {source[:120]}...")

Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION: What is LoRA and why is it used?


Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ANSWER:
The given context provides information regarding LoRA, a parameter-efficient fine-tuning method for deep neural networks that uses adaptive matrix multiplication techniques to reduce memory consumption while still achieving good performance. LoRA has been used as part of several research studies involving quantum computing applications such as the development of quantum algorithms for machine translation tasks, where it was found to be effective at improving accuracy compared to classical methods without sacrificing speed. Overall, LoRA offers a promising approach towards efficient and scalable quantum programming frameworks based on existing classical computer architectures.

SOURCE CHUNKS USED:
  [1] Q: What is LoRA?
A: LoRA stands for Low-Rank Adaptation. It is a parameter-efficient fine-tuning method that avoids upda...
  [2] Q: What are LoRA hyperparameters?
A: The key LoRA hyperparameters are rank (r), lora_alpha, lora_dropout, and target_mod...
  [3] Q: What is QLoRA?
A

Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



ANSWER:
The main differences between pre-training (which involves training a large pre-existing model with vast quantities of unstructured text) versus fine-tuning (wherein a pre-trained model is used as a starting point to improve performance towards a specific goal by adding tailored input data) can be summarized as follows:
1. Cost: While both methods require significant investment upfront, pre-training usually requires more expensive resources due to the need for relying heavily on huge datasets. In contrast, fine-tuning often comes down to a simple cost comparison when comparing the time required to generate output compared to the amount spent on acquiring new data.
2. Target audience: Pre-training tends to focus primarily on broader topics such as natural languages processing (NLP), while fine-tuning targets specific tasks within NLP domains.
3. Domain/Use Case Specificity: Pre-training generally works well across multiple applications and industries whereas fine-tuning is limit

## Cell 9 — Cell 9 — Test an Interview Question

As a final test, I'm using the RAG pipeline with an interview-related question from the knowledge document.

The idea is to see whether the system can retrieve the relevant information and use it to generate a useful answer for a question I might encounter while preparing for an LLM-related role.

In [ ]:
your_question = "How should I explain my transition from prompt engineering to LLM development?"

print(f"QUESTION: {your_question}")
print("=" * 65)

answer, sources = ask_rag(your_question)

print(f"\nANSWER:\n{answer}")
print(f"\nSOURCE CHUNKS USED:")
for i, source in enumerate(sources):
    print(f"  [{i+1}] {source[:120]}...")

Both `max_new_tokens` (=300) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION: How should I explain my transition from prompt engineering to LLM development?

ANSWER:
The given context provides insight into why the interviewer (Shruti Agrawal) wants to shift her focus towards LLM development after working with prompt engineering teams at Nvidia. The passage highlights her experience of being responsible for creating instruction response training datasets, annotating multimodal training data sets, and developing fine-tune workflows for deployable language modelling systems. This shows her proficiency in both prompt engineering and LLM development, which she has demonstrated by successfully completing two LLM project builds based upon the same knowledge base. Her ability to handle complex tasks such as data preparation, feature extraction, and machine learning algorithms demonstrates her versatility and adaptability when it comes to solving problems related to natural language processing. Overall, this information reinforces her qualifications as a skille

---
## Experiment Observation

The retrieval step generally returned relevant chunks for the questions I tested. However, the generated answers were not always fully grounded in those sources.

In some cases, TinyLlama added information that was not present in the retrieved context. This showed me that retrieving the right information is only one part of a RAG system; the generation step still needs to be evaluated for grounding and accuracy.

Takeaway

The main value of this experiment was seeing the RAG workflow at a lower level:

Document → Chunks → Embeddings → Vector Search → Retrieved Context → LLM → Answer